In [ ]:
import re
from pathlib import Path

SA_JSON = Path("/content/jlens-credentials/drive-sa.json")
DRIVE_HELPER = Path("/content/jlens-credentials/colab_drive.py")
WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

if SA_JSON.is_file():
    if not DRIVE_HELPER.is_file():
        raise RuntimeError(
            "Service-account credentials are present but colab_drive.py was not "
            "uploaded; use scripts/run_colab_notebook.sh for unattended CLI runs"
        )
    import runpy

    runpy.run_path(str(DRIVE_HELPER), run_name="__main__")
else:
    from google.colab import drive

    drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del (
    COMMIT_FILE,
    DIRTY_FILE,
    DRIVE_HELPER,
    REQUIREMENTS,
    SA_JSON,
    WHEEL_DIRECTORY,
    dirty_value,
    wheel,
    wheels,
)

# FLenQA linear probe assets

This notebook freezes a problem-level split, extracts final-token residual states, and saves one binary linear probe per transformer layer. It does not analyze J-Lens behavior, intervene on the model, or evaluate the held-out test split.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import transformers
from datasets import load_from_disk
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from experiments.flenqa_probe_jlens.analysis import validate_split
from experiments.flenqa_probe_jlens.constants import PROBE_CONFIG
from experiments.jlens_readout_sanity.constants import MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import (
    build_prompt_text,
    normalize_rows,
)
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.probing import (
    extract_probe_features,
    fit_binary_probe,
    probe_input_contract,
    save_probe_checkpoint,
)

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
# Reuse the original split while replacing the probe checkpoint and metadata.
SPLIT_PATH = context.checkpoints_dir / "flenqa-probe-assets" / "problem_split.json"
PROBE_PATH = ASSET_DIR / "probes.pt"
METADATA_PATH = ASSET_DIR / "metadata.json"
SPLIT_SEED = 1729
CONTEXT_SIZES = (250, 500)
C_GRID = (0.01, 0.1, 1.0, 10.0, 100.0)
ASSET_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

## Load FLenQA and identify underlying problems

`problem_id` is the normalized `global_sample_id`. The full-dataset validator checks that each underlying problem has 40 prompt variants.

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)

# One record per underlying problem, in the original split-generation order.
problem_metadata = {
    row.problem_id: {"problem_id": row.problem_id, "task": row.task, "label": row.label}
    for row in rows
}
problem_records = [
    problem_metadata[problem_id] for problem_id in sorted(problem_metadata)
]
problem_ids_by_task_label = Counter(
    (record["task"], record["label"]) for record in problem_records
)
problem_ids_by_task_label

## Create or load the fixed problem-level split

The split is stratified by task and gold label before any prompt variant is selected. Its assertions ensure that every variant inherits the same partition.

In [ ]:
split_fractions = {"train": 0.6, "validation": 0.2, "test": 0.2}
strata = [(record["task"], record["label"]) for record in problem_records]
if SPLIT_PATH.is_file():
    split_asset = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
else:
    train_records, heldout_records = train_test_split(
        problem_records,
        test_size=0.4,
        random_state=SPLIT_SEED,
        stratify=strata,
    )
    heldout_strata = [(record["task"], record["label"]) for record in heldout_records]
    validation_records, test_records = train_test_split(
        heldout_records,
        test_size=0.5,
        random_state=SPLIT_SEED,
        stratify=heldout_strata,
    )
    split_asset = {
        "format_version": 1,
        "seed": SPLIT_SEED,
        "fractions": split_fractions,
        "stratified_by": ["task", "label"],
        "problems": {
            "train": sorted(record["problem_id"] for record in train_records),
            "validation": sorted(record["problem_id"] for record in validation_records),
            "test": sorted(record["problem_id"] for record in test_records),
        },
        "problem_metadata": {
            str(record["problem_id"]): {
                "task": record["task"],
                "label": record["label"],
            }
            for record in problem_records
        },
    }
    SPLIT_PATH.write_text(
        json.dumps(split_asset, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

# Check assignments once, including leakage and agreement with dataset labels.
problem_to_split = validate_split(split_asset, rows)
{split: len(ids) for split, ids in split_asset["problems"].items()}

## Select 250- and 500-token training examples

Only train and validation problem IDs enter the probe workflow. The test partition remains untouched.

In [ ]:
selected_rows = [
    row
    for row in rows
    if row.ctx_size_declared in CONTEXT_SIZES
    and problem_to_split[row.problem_id] in {"train", "validation"}
]
selected_counts = Counter(
    (problem_to_split[row.problem_id], row.ctx_size_declared) for row in selected_rows
)
selected_counts

## Extract final-token residual states

For each prompt, the probed position is the last input token, immediately before answer generation. Hidden states are copied to CPU float32 before fitting. Input uses the same direct chat template as `generate_chat`: one user message, generation prefix enabled, thinking disabled, no truncation or extra BOS. The last input token is the end of that template prefix, not the end of the raw question. `hidden_states[l + 1]` is the block output except for the final entry, which Transformers replaces with the final normalized state. Retain this existing feature convention; never project the last probe as a pre-normalization J-Lens direction.

In [ ]:
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()
input_contract = probe_input_contract(tokenizer, config=PROBE_CONFIG)
num_layers = int(causal_lm.config.num_hidden_layers)

layer_features = [[] for _ in range(num_layers)]
example_labels = []
example_splits = []
for row in tqdm(selected_rows, desc="Extracting final-token states", unit="prompt"):
    prompt = build_prompt_text(
        task=row.task, question=row.question, mixin=row.mixin, rule=row.rule
    )
    features = extract_probe_features(causal_lm, tokenizer, prompt, config=PROBE_CONFIG)
    for layer_index, state in enumerate(features.states):
        layer_features[layer_index].append(state)
    example_labels.append(int(row.label))
    example_splits.append(problem_to_split[row.problem_id])

layer_features = [torch.stack(features) for features in layer_features]
labels = np.asarray(example_labels, dtype=np.int64)
example_splits = np.asarray(example_splits)
train_mask = example_splits == "train"
validation_mask = example_splits == "validation"
hidden_dim = int(layer_features[0].shape[1])
{"examples": len(selected_rows), "layers": num_layers, "hidden_dim": hidden_dim}

## Train one probe per layer

Features are centered by the train mean, without per-dimension scaling. Validation log loss selects the L2 regularization strength. Positive scores mean `True`; negative scores mean `False`.

In [ ]:
probe_layers = {}
for layer_index, features in enumerate(layer_features):
    probe_layers[layer_index] = fit_binary_probe(
        features[train_mask],
        labels[train_mask],
        features[validation_mask],
        labels[validation_mask],
        c_grid=C_GRID,
        seed=SPLIT_SEED,
    )

probe_summary = [
    {
        "layer": layer,
        "C": asset["C"],
        **{f"validation_{key}": value for key, value in asset["validation"].items()},
    }
    for layer, asset in probe_layers.items()
]
probe_summary[:3]

## Save frozen probe assets

The checkpoint contains tensors and metrics for every layer. The JSON sidecar keeps the split, extraction contract, and non-tensor metadata easy to inspect. Version 2 records the tokenizer/template fingerprint. Saving overwrites `probes.pt` and `metadata.json` in `flenqa-probe-assets` with compatible chat-format probes. The existing `problem_split.json` is reused without overwriting it. Rerun probe evaluation and J-Lens analysis after retraining.

In [ ]:
metadata = {
    **input_contract,
    "project_commit": PROJECT_COMMIT,
    "model_name": MODEL_NAME,
    "model_path": MODEL_PATH,
    "split_path": str(SPLIT_PATH),
    "split": split_asset,
    "split_seed": SPLIT_SEED,
    "context_sizes": list(CONTEXT_SIZES),
    "transformers_version": transformers.__version__,
    "label_convention": {"False": 0, "True": 1, "positive_score": "True"},
    "num_layers": num_layers,
    "hidden_dim": hidden_dim,
    "example_counts": {
        "train": int(train_mask.sum()),
        "validation": int(validation_mask.sum()),
    },
    "regularization_grid": list(C_GRID),
    "probe_metrics": {
        str(layer): {
            "C": asset["C"],
            "train": asset["train"],
            "validation": asset["validation"],
        }
        for layer, asset in probe_layers.items()
    },
}
save_probe_checkpoint(
    PROBE_PATH,
    probe_layers,
    metadata=metadata,
    metadata_path=METADATA_PATH,
)
print(f"Saved split: {SPLIT_PATH}")
print(f"Saved probes: {PROBE_PATH}")
print(f"Saved metadata: {METADATA_PATH}")
print(
    {
        "train_examples": int(train_mask.sum()),
        "validation_examples": int(validation_mask.sum()),
        "layers": num_layers,
    }
)